In [ ]:
import copy
from typing import TypeVar

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import pyref.fitting as fit
import scipy
import sigfig
from refnx.analysis import Interval, Parameter, Parameters


In [ ]:
#Chi Squared Functions (probably me messing around tbh)

def fitDist_val(y, dy, expected):
    return (y - expected) / dy #use expected again if there's no uncertainty aka dy value

def chiSquare_val( y, dy, expected):
    dist = fitDist_val(y, dy, expected)
    return np.dot(dist, dist) #dotting it takes care of squaring + summing

#def chiSquare_Red_val(y, dy, fittingParams,expected):
#    dist = fitDist_val(y, dy, expected)
#    dof = (len(y) - fittingParams)
#    return np.dot(dist, dist) / dof

def chiSq_PDF(chiSq, DOF, labels):
    _5percent = scipy.stats.chi2.ppf(0.05, DOF)
    _95percent = scipy.stats.chi2.ppf(0.95, DOF)
    x_region = np.linspace(scipy.stats.chi2.ppf(0.00001, DOF), scipy.stats.chi2.ppf(0.9999, DOF), 1000)
    full_PDF = scipy.stats.chi2.pdf(x_region, DOF)

    #plot it
    plt.plot(x_region, full_PDF, 'r', alpha = 0.8, label = r'$\chi^2$ PDF')
    plt.plot((_5percent, _5percent), (0, 1.2* np.max(full_PDF)), 'b', label = fr'5% probability: $\chi^2$ = {np.round(_5percent, 4)}')
    plt.plot((_95percent, _95percent), (0, 1.2* np.max(full_PDF)), 'b', label = fr'95% probability: $\chi^2$ = {np.round(_95percent, 4)}')
    #if there's multiple chi sqs to plot at once
    for i in range(len(chiSq)):
        plt.plot((chiSq[i],chiSq[i]), (0,1.2*np.max(full_PDF)), 'k', label = labels[i])
    plt.legend(loc='upper right')
    plt.title(r"$\chi^2$ Probability Distribution Function")
    plt.xlabel("Chi Squared Value")
    plt.ylabel("Probability")
    plt.grid()
    plt.tight_layout()
    plt.show()


In [ ]:
#Fit Parameter Functions
def all_parameters(parameter_group):
    if isinstance(parameter_group, Parameters): #returns true when the first thing is of the second's object type
        params = []
        for param in parameter_group:
            params.extend(all_parameters(param))
        return params
    else:
        return [parameter_group]


def print_parameters(obj):
    """Print all parameters in the object."""
    param_list = all_parameters(obj.parameters)
    print(f"Number of parameters: {len(param_list)}")
    print(f"Varying parameters: {len(obj.varying_parameters())}")
    print("-" * 82)
    for param in param_list:
        vary_status = "✓" if param.vary else "✗"
        color = (
            "\033[94m" if param.vary else "\033[91m"
        )  # Light blue for vary=True, red for vary=False
        reset_color = "\033[0m"  # Reset to default color
        if param.stderr and param.stderr != 0:
            stderr = sigfig.round(param.stderr, sigfigs=1)
            val = round(param.value, -int(np.floor(np.log10(param.stderr))))
            val_str = f"{val:10.3f}"
            stderr_str = f"{stderr:g}"  # Remove trailing zeros
        else:
            stderr_str = "0"
            val_str = (
                f"{param.value:10.3f}"
                if param.name != "bkg"
                else f"{param.value:10.3g}"
            )

        print(
            f"|{color}{vary_status} | {param.name:30} | {val_str} ± {stderr_str:>8} | {param.bounds!s:18}{reset_color} |"
        )
    print("-" * 82)


In [ ]:
#import excel of merged data
df = pd.read_csv("HiRes_BigRegion_ExcelMerge_CSV.csv", skiprows = 0)#, sep=r"\s+")
#df.plot(
#    x = "q(deg)",
#    y="I_merged(a.u.)",
#    yerr="Sigma_I(a.u.)",
#    )
#df.rename(columns={'IncidentAngle(deg)':'Q','intU':'Sigma_I(a.u.)'}, inplace=True)
df.head()


In [ ]:
#convert angle to q and rename columns
beam_energy = 8.04
wavelength = 1.54209 #angstrom #https://www.kmlabs.com/en/wavelength-to-photon-energy-calculator
radians = np.array(df["IncidentAngle(deg)"]) * np.pi / 180
q_range = np.sin(radians) * 4 * np.pi / wavelength

df["IncidentAngle(deg)"] = q_range
df.rename(columns={'IncidentAngle(deg)':'Q', "Intensity(a.u.)":"r", "Sigma_I(a.u.)":"int_u"}, inplace=True)
df.head()


In [ ]:
df["int_u"] /= df["r"].max()
df["r"] /= df["r"].max()
df["int_u"] = .1 * df["r"]
df.describe()


In [ ]:
data_tuple = (df["Q"].to_numpy(), df["r"].to_numpy(), df["int_u"].to_numpy())
#data_tuple


In [ ]:
#reformat data then plot it
data = fit.ReflectDataset( #reformats automatically for xray data #to_reflect_dataset
    data=data_tuple
)
data.plot()
plt.yscale("log")
#ax_top.set_title("Reflectivity Data")
#ax_bottom.set_ylabel("Anisotropy")

# Just some aesthetic adjustments
#fig = plt.gcf()
#ax_top.legend(["s-polarized", "p-polarized"])
#fig.set_size_inches(8, 6)
#fig.align_labels()
#plt.tight_layout()
#plt.show()


In [ ]:
#Create slab simulation
energy = 8.04 * 10**3  # Energy in eV
vacuum_generator = fit.MaterialSLD(
    formula="",  # No formula for vacuum
    density=0,
    energy=energy,
    name="Vacuum",  # Name of the material
)
vacuum = vacuum_generator(
    thick=0,  # For the vacuum layer, we can set thickness to 0
    rough=0,  # Roughness in Angstroms
)

ZNPC = "C32H16N8Zn1"  # Stoichiometry fir Zinc (III) phthalocyanine
bulk_generator = fit.MaterialSLD(
    formula=ZNPC,
    density=1.6,  # Density of Zinc phthalocyanine
    energy=energy,  # Energy in eV
    name="Zinc Phthalocyanine",
)

bulk = bulk_generator(
    thick=170, #170
    rough=10,  # Roughness in Angstroms this is a good guess
)

oxide_generator = fit.MaterialSLD(
    formula="SiO2",  # Silicon dioxide
    density=2.23,  # Density of SiO2
    energy=energy,  # Energy in eV
    name="Silicon Dioxide",
)
oxide = oxide_generator(
    thick=8.8,  # Thickness in Angstroms
    rough=4,  # Roughness in Angstroms
)

interface_generator = fit.MaterialSLD(
    formula=ZNPC,
    density=1.6,
    energy=energy,
    name="interface",
)
interface = interface_generator(
    thick=10,
    rough=2,
)
surface_generator = fit.MaterialSLD(
    formula=ZNPC,
    density=1.6,
    energy=energy,
    name="surface",
)
surface = surface_generator(
    thick=10,
    rough=2,
)



substrate_generator = fit.MaterialSLD(
    formula="Si",  # Silicon
    density=2.3,  # Density of Silicon
    energy=energy,  # Energy in eV
    name="Silicon Substrate",
)
# The substrate layer has transparent boundary conditions below the slab, so we set the
# thickness to 0 to signal that this is a boundary slab that extends infinitely.
substrate = substrate_generator(
    thick=0,  # For the vacuum and substrate layers, we can set thickness to 0
    rough=1.5,  # This tends to be a good guess for silicon substrates
)

# Hold constant the parameters that we want to fix during the fit
vacuum.thick.setp(vary=False)
vacuum.rough.setp(vary=False)
vacuum.sld.density.setp(vary=False)

# Recentered on the differential-evolution/MCMC posterior medians. Bounds widened
# wherever the posterior sat within ~1-2 stderr of a bound (a sign the hard prior was
# truncating the walk, not that the value was truly at that limit).
oxide.thick.setp(value=8.06203, vary=True, bounds=(4, 14))  # was (6, 12): ~2.3 sigma from lower bound
oxide.rough.setp(value=7.85392, vary=True, bounds=(0, 10))
oxide.sld.density.setp(value=2.2, vary=True, bounds=(2, 2.4))  # was (2, 2.4): ~1.5 sigma from lower bound
#oxide.sld.density.setp(vary=False, value=2.3)

# Silicon Substrate_thick is the semi-infinite backing layer -- it must stay at 0 and
# can never actually move. It was previously vary=True with zero-width bounds (0, 0),
# which broke MCMC diagnostics (obj.corner()) since a "varying" parameter that can
# never move produces a zero-variance chain column.
substrate.thick.setp(vary=False, value=0)
substrate.rough.setp(value=1.20002, vary=False, bounds=(0.5, 2.0))  # was (1.0, 1.7): ~1 sigma from lower bound
substrate.sld.density.setp(value=2.38326, vary=False, bounds=(2.2, 3))  # was (2.3, 2.4): ~0.9 sigma from upper bound

interface.thick.setp(value=19.0594, vary=True, bounds=(0, 25))  # was (0, 20): ~1 sigma from upper bound
interface.rough.setp(value=11.5823, vary=True, bounds=(0, 15))  # was (0, 12): ~1 sigma from upper bound
interface.sld.density.setp(value=0.629554, vary=True, bounds=(0.2, 0.7))

surface.thick.setp(value=8, vary=True, bounds=(0, 25))  # was (0, 20): ~0.85 sigma from upper bound
surface.rough.setp(value=6, vary=True, bounds=(0, 15))
surface.sld.density.setp(value=2.1, vary=True, bounds=(1.8, 3))

bulk.thick.setp(value=114.316, vary=True, bounds=(90, 200))
bulk.rough.setp(value=9, vary=True, bounds=(5, 30))  # was (0, 30): ~3.3 sigma from upper bound
bulk.sld.density.setp(value=1.619, vary=False, bounds=(1, 2))

# Now we can create a multilayer structure using these generators, and the elipsometry
# results for thickness and roughness. To build the slab, we use the logical OR "|"
# operator to combine the layers.
slab = vacuum |surface| bulk |interface| oxide | substrate
slab.name = "Single layer Zinc Phthalocyanine on Silicon Substrate"
print(slab)
slab.plot()


In [ ]:
#plot the simulation (no data fitting yet)
# set the ticks to point inwards
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["ytick.direction"] = "in"

# For the fitting, we need to define a model.

s_pol_function = fit.ReflectModel(
    structure=slab,  # The slab we just created
    energy=energy,  # Energy in eV
    pol="s",  # S-polarized light
    name="Zinc Phthalocyanine Reflectivity Model",  # Name of the model
)
p_pol_function = fit.ReflectModel(
    structure=slab,  # The slab we just created
    pol="p",  # P-polarized light
    energy=energy,  # Energy in eV
    name="Zinc Phthalocyanine Reflectivity Model",  # Name of the model
)
q = np.linspace(0.01, 0.25, 100)
s_pol_reflectivity = s_pol_function(q)  # S-polarized reflectivity
p_pol_reflectivity = p_pol_function(q)  # P-polarized reflectivity

# Now we can plot the reflectivity for both polarizations.
fig, ax = plt.subplots()
ax.plot(q, s_pol_reflectivity, label="S-Polarized Reflectivity")
ax.plot(q, p_pol_reflectivity, label="P-Polarized Reflectivity", linestyle="--")
ax.set_xlabel(r"Momentum Transfer $(1/\AA)$")
ax.set_ylabel("Reflectivity")
ax.set_title("Reflectivity vs Momentum Transfer")
ax.set_yscale("log")  # Reflectivity is often plotted on a logarithmic scale
ax.legend()
# just some aesthetic adjustments
ax.set_xlim(0, 0.25)  # Set x-axis limits
ax.set_ylim(1e-9, 1)


In [ ]:
model = fit.ReflectModel( #expected model based on earlier simulation
    structure=slab,  # The slab we just created
    energy=energy,  # Energy in eV
    pol="s",  # S and P-polarized light
    name="Zinc Phthalocyanine Reflectivity Model",  # Name of the model
)
print(model.parameters)


In [ ]:
bkg = data.y.min()

model.bkg.setp(value=bkg, vary=False)  # Set the background to a fixed value
# we will also set the scale and theta offset parameters to vary but arround a fixed value
model.scale_s.setp(vary=False, value=0.987, bounds=(0.6, 1.4))
model.scale_p.setp(vary=False, value=1.0, bounds=(0.8, 1.2))
# theta offset parameters should be small or they will cause issues with the fit
model.theta_offset_s.setp(vary=True, value=0.00412263, bounds=(-0.8, 0.8))
model.theta_offset_p.setp(vary=False, value=0.0, bounds=(-0.8, 0.8))

model.dq.setp(vary=False, value=0.0)


In [ ]:
obj = fit.Objective( #AnisotropyObjective
    data=data,  # The data we just created
    model=model,  # The model we just created
    name="Zinc Phthalocyanine Reflectivity Objective",  # Name of the objective
    transform=fit.Transform("logY"),
)
# we also want to apply this little modification to the logp of the system.
lpe = fit.LogpExtra(obj)
obj.logp_extra = lpe
# This ensures that the roughness and thickness parameters are physical.

fit.CurveFitter(obj).fit(
    "differential_evolution",
    target = "nlpost",
)

print(obj.varying_parameters())


In [ ]:
obj.plot()
plt.show()
obj.model.structure.plot()
plt.show()


In [ ]:
print(obj.varying_parameters())


In [ ]:
obj.model.structure.plot()


# These are probably what you want


In [ ]:
plt.rcParams['figure.dpi'] = 400 #SET DPI


In [ ]:
#mc-lookin fit
fitter = fit.CurveFitter(obj, walkers_per_param=20)
fitter.initialise("jitter")
fitter.sample(
    1000, skip_check=True
)  # this will likely have a large initial condition number, so we skip the check and just sample
nlpost = -fitter.logpost
plt.plot(nlpost, c="k", lw=0.2, alpha=0.5)
plt.show()


In [ ]:
import corner as corner_pkg

# Silicon Substrate_thick is fixed at 0 with zero-width bounds (it's the semi-infinite
# substrate boundary layer, so it can never actually vary), which leaves its MCMC chain
# column with no dynamic range and breaks obj.corner()'s automatic range detection.
# Drop any such zero-variance columns before plotting.
var_pars = obj.varying_parameters()
chain = np.array([p.chain for p in var_pars]).reshape(len(var_pars), -1).T
labels = np.array([p.name for p in var_pars])

varies = chain.std(axis=0) > 0
if not np.all(varies):
    dropped = ", ".join(labels[~varies])
    print(f"Dropping parameter(s) with no dynamic range from corner plot: {dropped}")

corner_pkg.corner(chain[:, varies], labels=labels[varies], quantiles=[0.16, 0.5, 0.84])


In [ ]:
print(obj.varying_parameters())


In [ ]:
#this may have been me experimenting
y_model = obj.model(obj.data.x)
y_data, y_data_err, y_model_log = obj._data_transform(y_model)

chi_test = chiSquare_val(df["r"], y_data_err, y_model)
chiSq_PDF([chi_test], len(y_data)-12, [fr"$\chi^2$ = {chi_test:.4f}"])
